# NOTEBOOK 6: URGENCY SCORING & PRIORITIZATION

**Goal**: Score each complaint 0-100 and assign to priority buckets (P1/P2/P3/P4)

After Notebook 5 labels complaints as: positive / neutral / negative / critical

This notebook combines 4 factors with specific weights to create an urgency score and priority tier.

In [26]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

✅ Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.3.5


## Configuration: Weights, Thresholds & Priority Rules

In [27]:
# ── FACTOR WEIGHTS ──────────────────────────────────────
# How important is each factor in the urgency score?
# These weights MUST sum to 1.0

WEIGHTS = {
    'sentiment': 0.40,    # 40% - AI model sentiment label (most reliable signal)
    'keywords': 0.25,     # 25% - Emergency keywords (safety net if model fails)
    'confidence': 0.20,   # 20% - Model confidence in its prediction
    'recency': 0.15       # 15% - How recently complaint was filed
}

# Verify weights sum to 1.0
total_weight = sum(WEIGHTS.values())
print(f'Total weight: {total_weight}')
assert abs(total_weight - 1.0) < 0.001, f'Weights must sum to 1.0, got {total_weight}'
print('✅ Weights validated\n')

# ── SENTIMENT SCORING (0-100) ──────────────────────────
# Maps sentiment labels to base scores
SENTIMENT_SCORES = {
    'critical': 100,
    'negative': 60,
    'neutral': 20,
    'positive': 0
}

print('Sentiment Scores:')
for label, score in SENTIMENT_SCORES.items():
    print(f'  {label:12} → {score:3d} points')

# ── PRIORITY TIERS ──────────────────────────────────────
# Score ranges and response time SLAs
PRIORITY_TIERS = {
    'P1': {'score_min': 80, 'score_max': 100, 'sla': '2 hours'},
    'P2': {'score_min': 60, 'score_max': 79,  'sla': '24 hours'},
    'P3': {'score_min': 40, 'score_max': 59,  'sla': '3 days'},
    'P4': {'score_min': 0,  'score_max': 39,  'sla': '7 days'}
}

print('\nPriority Tiers:')
for tier, info in PRIORITY_TIERS.items():
    print(f'  {tier} ({info["score_min"]:2d}-{info["score_max"]:2d}) → Respond within {info["sla"]}')

Total weight: 1.0
✅ Weights validated

Sentiment Scores:
  critical     → 100 points
  negative     →  60 points
  neutral      →  20 points
  positive     →   0 points

Priority Tiers:
  P1 (80-100) → Respond within 2 hours
  P2 (60-79) → Respond within 24 hours
  P3 (40-59) → Respond within 3 days
  P4 ( 0-39) → Respond within 7 days


## Load Sentiment Model Output from Notebook 5

In [28]:
# Define paths
BASE_DIR = Path('../')
DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUTPUTS_DIR = BASE_DIR / 'outputs'

# Create outputs directory if needed
OUTPUTS_DIR.mkdir(exist_ok=True)

# Load the processed data with sentiment labels from Notebook 5
try:
    df = pd.read_csv(DATA_DIR / 'processed' / 'grievance_processed.csv')
    print(f'✅ Loaded processed data: {df.shape[0]:,} rows × {df.shape[1]} columns')
except FileNotFoundError:
    print('⚠️  grievance_processed.csv not found, using cleaned data instead')
    df = pd.read_csv(DATA_DIR / 'cleaned' / 'grievance_cleaned.csv')
    print(f'✅ Loaded cleaned data: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Display first few rows
print('\nFirst 2 rows:')
print(df.head(2))

# Display columns
print(f'\nColumns ({len(df.columns)}):') 
print(df.columns.tolist())

✅ Loaded processed data: 12,387 rows × 20 columns

First 2 rows:
   Unique Key         Created Date          Closed Date  \
0    32310363  2015-12-31 23:59:45  2016-01-01 00:55:15   
1    32309934  2015-12-31 23:59:44  2016-01-01 01:26:57   

                       Agency Name           Complaint Type        Descriptor  \
0  New York City Police Department  Noise - Street/Sidewalk  Loud Music/Party   
1  New York City Police Department         Blocked Driveway         No Access   

     Location Type    Borough  Status  \
0  Street/Sidewalk  MANHATTAN  Closed   
1  Street/Sidewalk     QUEENS  Closed   

                              Resolution Description  resolution_hours  \
0  The Police Department responded and upon arriv...          0.925000   
1  The Police Department responded to the complai...          1.453611   

   hour_created day_of_week      closed_date_fmt resolution_hours_fmt  \
0            23    Thursday  2016-01-01 00:55:15                <1 hr   
1            23    T

## Define Emergency Keyword Tiers

In [29]:
# Emergency keywords tier system - acts as safety net if model misclassifies
EMERGENCY_KEYWORDS = {
    'tier_1': {
        'keywords': ['fire', 'collapse', 'trapped', 'explosion', 'drown', 'suffocation'],
        'points': 40,
        'description': 'Life-threatening/immediate danger'
    },
    'tier_2': {
        'keywords': ['urgent', 'emergency', 'danger', 'live wire', 'electrocute', 'hazard'],
        'points': 25,
        'description': 'High risk/serious concern'
    },
    'tier_3': {
        'keywords': ['broken', 'garbage', 'blocked', 'not working', 'flooding', 'leak', 'hazardous'],
        'points': 10,
        'description': 'Infrastructure/service issue'
    }
}

print('Emergency Keyword Tiers:')
for tier, info in EMERGENCY_KEYWORDS.items():
    print(f'\n{tier.upper()} ({info["points"]} pts): {info["description"]}')
    print(f'  Keywords: {info["keywords"]}')

Emergency Keyword Tiers:

TIER_1 (40 pts): Life-threatening/immediate danger
  Keywords: ['fire', 'collapse', 'trapped', 'explosion', 'drown', 'suffocation']

TIER_2 (25 pts): High risk/serious concern
  Keywords: ['urgent', 'emergency', 'danger', 'live wire', 'electrocute', 'hazard']

TIER_3 (10 pts): Infrastructure/service issue
  Keywords: ['broken', 'garbage', 'blocked', 'not working', 'flooding', 'leak', 'hazardous']


## Build Scoring Functions

In [30]:
def get_sentiment_score(sentiment_label):
    """Convert sentiment label to 0-100 score."""
    if pd.isna(sentiment_label):
        return 20  # Default to neutral if missing
    sentiment_label = str(sentiment_label).lower().strip()
    return SENTIMENT_SCORES.get(sentiment_label, 20)


def get_keyword_score(text):
    """Check for emergency keywords and return score (0-100)."""
    if pd.isna(text):
        return 0
    
    text_lower = str(text).lower()
    score = 0
    
    # Check Tier 1 keywords (40 pts)
    for keyword in EMERGENCY_KEYWORDS['tier_1']['keywords']:
        if keyword in text_lower:
            return EMERGENCY_KEYWORDS['tier_1']['points']  # Cap at tier 1
    
    # Check Tier 2 keywords (25 pts)
    for keyword in EMERGENCY_KEYWORDS['tier_2']['keywords']:
        if keyword in text_lower:
            score = max(score, EMERGENCY_KEYWORDS['tier_2']['points'])
    
    # Check Tier 3 keywords (10 pts)
    if score == 0:  # Only if no tier 1 or 2 matched
        for keyword in EMERGENCY_KEYWORDS['tier_3']['keywords']:
            if keyword in text_lower:
                score = max(score, EMERGENCY_KEYWORDS['tier_3']['points'])
    
    return score


def get_confidence_score(confidence):
    """Convert model confidence (0-1 or 0-100) to score.
    
    Logic: 99% confident → ~99, 51% confident → ~51
    """
    if pd.isna(confidence) or confidence is None:
        return 50  # Default to medium confidence if missing
    
    confidence = float(confidence)
    # If confidence is 0-1, convert to 0-100
    if confidence <= 1.0:
        confidence = confidence * 100
    
    # Cap between 0 and 100
    return min(100, max(0, confidence))


def get_recency_score(created_date, reference_date=None):
    """Calculate recency score (0-100).
    
    Today = 100, 30 days ago = 0
    """
    if pd.isna(created_date):
        return 0  # Very old if unknown
    
    if reference_date is None:
        reference_date = pd.Timestamp.now()
    else:
        reference_date = pd.Timestamp(reference_date)
    
    created_date = pd.Timestamp(created_date)
    
    # Days since complaint was filed
    days_old = (reference_date - created_date).days
    
    # Map: 0 days = 100, 30 days = 0 (linear)
    # Negative values (future dates) → 100
    if days_old < 0:
        return 100
    
    recency_score = max(0, 100 - (days_old / 30 * 100))
    return recency_score


def calculate_urgency_score(row, reference_date=None):
    """
    Main scoring function combining all 4 factors.
    
    Formula:
    urgency = (sentiment × 0.40) + (keywords × 0.25) + (confidence × 0.20) + (recency × 0.15)
    
    Args:
        row: pandas Series with complaint data
        reference_date: Date to calculate recency against (default: today)
    
    Returns:
        dict with scores for each factor and final urgency score (0-100)
    """
    # Get scores for each factor
    sentiment_score = get_sentiment_score(row.get('sentiment_label', 'neutral'))
    keyword_score = get_keyword_score(row.get('combined_text', '') or row.get('clean_text', ''))
    confidence_score = get_confidence_score(row.get('model_confidence', None))
    recency_score = get_recency_score(row.get('Created Date', None), reference_date)
    
    # Calculate weighted sum
    urgency = (
        (sentiment_score * WEIGHTS['sentiment']) +
        (keyword_score * WEIGHTS['keywords']) +
        (confidence_score * WEIGHTS['confidence']) +
        (recency_score * WEIGHTS['recency'])
    )
    
    # Cap at 100
    urgency = min(100, max(0, urgency))
    
    return {
        'sentiment_score': sentiment_score,
        'keyword_score': keyword_score,
        'confidence_score': confidence_score,
        'recency_score': recency_score,
        'urgency_score': urgency
    }


def get_priority_tier(urgency_score):
    """Convert urgency score to priority tier (P1/P2/P3/P4)."""
    for tier, info in PRIORITY_TIERS.items():
        if info['score_min'] <= urgency_score <= info['score_max']:
            return tier
    return 'P4'  # Default to lowest priority


print('✅ All scoring functions defined successfully')

✅ All scoring functions defined successfully


## Test on Sample Complaints

In [31]:
# Take first 6 rows for testing
test_df = df.head(6).copy()

# Add placeholder sentiment labels if column doesn't exist
if 'sentiment_label' not in test_df.columns:
    # For testing, assign random sentiments
    sentiments = ['positive', 'neutral', 'negative', 'critical', 'neutral', 'negative']
    test_df['sentiment_label'] = sentiments
    print('⚠️  Added placeholder sentiment labels for testing')

# Add placeholder confidence if doesn't exist
if 'model_confidence' not in test_df.columns:
    confidences = [0.85, 0.92, 0.78, 0.95, 0.88, 0.81]
    test_df['model_confidence'] = confidences
    print('⚠️  Added placeholder confidence scores for testing')

# Calculate urgency scores for test samples
print('SAMPLE URGENCY SCORING TEST')

for idx, (i, row) in enumerate(test_df.iterrows(), 1):
    scores = calculate_urgency_score(row)
    priority = get_priority_tier(scores['urgency_score'])
    
    print(f'\n📋 COMPLAINT #{idx} (ID: {row.get("Unique Key", "N/A")})')
    print(f'   Text: {(row.get("clean_text", "")[:60])}...')
    print(f'   Sentiment: {row.get("sentiment_label", "N/A")}')
    print(f'   ─────────────────────────────────')
    print(f'   Sentiment Score    : {scores["sentiment_score"]:6.2f} (40% weight)')
    print(f'   Keyword Score      : {scores["keyword_score"]:6.2f} (25% weight)')
    print(f'   Confidence Score   : {scores["confidence_score"]:6.2f} (20% weight)')
    print(f'   Recency Score      : {scores["recency_score"]:6.2f} (15% weight)')
    print(f'   ═════════════════════════════════')
    print(f'   🎯 URGENCY SCORE   : {scores["urgency_score"]:6.2f}')
    print(f'   🚨 PRIORITY TIER   : {priority} (SLA: {PRIORITY_TIERS[priority]["sla"]})')

print('✅ Sample test completed successfully')

⚠️  Added placeholder sentiment labels for testing
⚠️  Added placeholder confidence scores for testing
SAMPLE URGENCY SCORING TEST

📋 COMPLAINT #1 (ID: 32310363)
   Text: noise street sidewalk loud music party upon responsible gone...
   Sentiment: positive
   ─────────────────────────────────
   Sentiment Score    :   0.00 (40% weight)
   Keyword Score      :   0.00 (25% weight)
   Confidence Score   :  85.00 (20% weight)
   Recency Score      :   0.00 (15% weight)
   ═════════════════════════════════
   🎯 URGENCY SCORE   :  17.00
   🚨 PRIORITY TIER   : P4 (SLA: 7 days)

📋 COMPLAINT #2 (ID: 32309934)
   Text: blocked driveway access...
   Sentiment: neutral
   ─────────────────────────────────
   Sentiment Score    :  20.00 (40% weight)
   Keyword Score      :  10.00 (25% weight)
   Confidence Score   :  92.00 (20% weight)
   Recency Score      :   0.00 (15% weight)
   ═════════════════════════════════
   🎯 URGENCY SCORE   :  28.90
   🚨 PRIORITY TIER   : P4 (SLA: 7 days)

📋 COMPLAINT 

## Score All Complaints

In [32]:
print(f'Starting to score {len(df):,} complaints...')
print('This may take a minute or two...')
print()

# Apply scoring to all rows
scores_list = []
for idx, row in df.iterrows():
    if (idx + 1) % 2000 == 0:
        print(f'  Processed {idx + 1:,} / {len(df):,} complaints...')
    scores_list.append(calculate_urgency_score(row))

# Convert list of dicts to DataFrame
scores_df = pd.DataFrame(scores_list)

# Combine with original data
df_scored = pd.concat([df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)

# Add priority tiers
df_scored['priority_tier'] = df_scored['urgency_score'].apply(get_priority_tier)

# Sort by urgency score (highest first)
df_scored_sorted = df_scored.sort_values('urgency_score', ascending=False).reset_index(drop=True)

print(f'\n✅ Scoring completed!')
print(f'   Total complaints scored: {len(df_scored_sorted):,}')
print(f'   Columns: {len(df_scored_sorted.columns)}')
print(f'\nFirst 5 rows (highest urgency):')
print(df_scored_sorted[['Unique Key', 'urgency_score', 'priority_tier', 
                          'sentiment_score', 'keyword_score', 'confidence_score']].head())

Starting to score 12,387 complaints...
This may take a minute or two...

  Processed 2,000 / 12,387 complaints...
  Processed 4,000 / 12,387 complaints...
  Processed 6,000 / 12,387 complaints...
  Processed 8,000 / 12,387 complaints...
  Processed 10,000 / 12,387 complaints...
  Processed 12,000 / 12,387 complaints...

✅ Scoring completed!
   Total complaints scored: 12,387
   Columns: 26

First 5 rows (highest urgency):
   Unique Key  urgency_score priority_tier  sentiment_score  keyword_score  \
0    29617561          24.25            P4               20             25   
1    31119225          24.25            P4               20             25   
2    31117830          24.25            P4               20             25   
3    31123281          24.25            P4               20             25   
4    31116976          24.25            P4               20             25   

   confidence_score  
0                50  
1                50  
2                50  
3                

## Save Results

In [33]:
# Define output file path
output_csv = OUTPUTS_DIR / 'grievance_with_urgency.csv'

# Save scored dataset (sorted by urgency)
df_scored_sorted.to_csv(output_csv, index=False)
print(f'✅ Saved: {output_csv}')
print(f'   File size: {output_csv.stat().st_size / 1024 / 1024:.2f} MB')
print(f'   Rows: {len(df_scored_sorted):,}')
print(f'   Columns: {len(df_scored_sorted.columns)}')

✅ Saved: ..\outputs\grievance_with_urgency.csv
   File size: 6.89 MB
   Rows: 12,387
   Columns: 26


## Priority Distribution Report

In [34]:
print('PRIORITY DISTRIBUTION REPORT')

# Count complaints by priority tier
priority_dist = df_scored_sorted['priority_tier'].value_counts().sort_index()

print('\nComplaints by Priority Tier:')
total_complaints = len(df_scored_sorted)
for tier in ['P1', 'P2', 'P3', 'P4']:
    count = priority_dist.get(tier, 0)
    pct = (count / total_complaints) * 100
    sla = PRIORITY_TIERS[tier]['sla']
    print(f'  {tier} ({PRIORITY_TIERS[tier]["score_min"]:2d}-{PRIORITY_TIERS[tier]["score_max"]:2d}): {count:6,} complaints ({pct:5.2f}%) → {sla}')

print(f'  ─────────────────────────────────')
print(f'  TOTAL                : {total_complaints:6,} complaints')

# Statistics on urgency scores
print('\nUrgency Score Statistics:')
print(f'  Min Score  : {df_scored_sorted["urgency_score"].min():.2f}')
print(f'  Max Score  : {df_scored_sorted["urgency_score"].max():.2f}')
print(f'  Mean Score : {df_scored_sorted["urgency_score"].mean():.2f}')
print(f'  Median Score : {df_scored_sorted["urgency_score"].median():.2f}')
print(f'  Std Dev    : {df_scored_sorted["urgency_score"].std():.2f}')

# Top 10 most urgent complaints
print('\nTop 10 Most Urgent Complaints:')
top_10 = df_scored_sorted.head(10)[['Unique Key', 'priority_tier', 'urgency_score', 
                                      'Complaint Type', 'clean_text']].copy()
top_10['clean_text'] = top_10['clean_text'].str[:50]
print(top_10.to_string(index=False))

PRIORITY DISTRIBUTION REPORT

Complaints by Priority Tier:
  P1 (80-100):      0 complaints ( 0.00%) → 2 hours
  P2 (60-79):      0 complaints ( 0.00%) → 24 hours
  P3 (40-59):      0 complaints ( 0.00%) → 3 days
  P4 ( 0-39): 12,387 complaints (100.00%) → 7 days
  ─────────────────────────────────
  TOTAL                : 12,387 complaints

Urgency Score Statistics:
  Min Score  : 18.00
  Max Score  : 24.25
  Mean Score : 20.10
  Median Score : 20.50
  Std Dev    : 2.31

Top 10 Most Urgent Complaints:
 Unique Key priority_tier  urgency_score     Complaint Type                                         clean_text
   29617561            P4          24.25    Illegal Parking illegal parking blocked hydrant forwarded non emer
   31119225            P4          24.25   Blocked Driveway blocked driveway access forwarded non emergency re
   31117830            P4          24.25    Illegal Parking illegal parking double parked blocking traffic for
   31123281            P4          24.25    Ille

## Save Configuration for Production

In [35]:
# Prepare config dictionary
config = {
    'created_at': datetime.now().isoformat(),
    'description': 'Complaints urgency scoring and prioritization configuration',
    'version': '1.0',
    'weights': WEIGHTS,
    'sentiment_scores': SENTIMENT_SCORES,
    'priority_tiers': {
        tier: {
            'score_min': info['score_min'],
            'score_max': info['score_max'],
            'sla': info['sla']
        }
        for tier, info in PRIORITY_TIERS.items()
    },
    'emergency_keywords': EMERGENCY_KEYWORDS,
    'formula': 'urgency = (sentiment × 0.40) + (keywords × 0.25) + (confidence × 0.20) + (recency × 0.15)',
    'statistics': {
        'total_complaints': int(len(df_scored_sorted)),
        'urgency_score_min': float(df_scored_sorted['urgency_score'].min()),
        'urgency_score_max': float(df_scored_sorted['urgency_score'].max()),
        'urgency_score_mean': float(df_scored_sorted['urgency_score'].mean()),
        'urgency_score_median': float(df_scored_sorted['urgency_score'].median()),
        'priority_distribution': {
            tier: int(priority_dist.get(tier, 0)) 
            for tier in ['P1', 'P2', 'P3', 'P4']
        }
    }
}

# Save config as JSON
config_path = OUTPUTS_DIR / 'urgency_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ Saved configuration: {config_path}')
print('\nConfiguration saved:')
print(json.dumps(config, indent=2)[:500] + '...')

✅ Saved configuration: ..\outputs\urgency_config.json

Configuration saved:
{
  "created_at": "2026-03-31T03:03:07.401528",
  "description": "Complaints urgency scoring and prioritization configuration",
  "version": "1.0",
  "weights": {
    "sentiment": 0.4,
    "keywords": 0.25,
    "confidence": 0.2,
    "recency": 0.15
  },
  "sentiment_scores": {
    "critical": 100,
    "negative": 60,
    "neutral": 20,
    "positive": 0
  },
  "priority_tiers": {
    "P1": {
      "score_min": 80,
      "score_max": 100,
      "sla": "2 hours"
    },
    "P2": {
      "score_mi...


## Live Scoring Function for New Complaints
    Args:
        complaint_text (str): The complaint text
        sentiment_label (str): Sentiment from AI model (positive/neutral/negative/critical)
        model_confidence (float): Model confidence (0-1 or 0-100)
        created_date (str or datetime): When complaint was filed (default: now)
    
    Returns:
        dict with urgency_score, priority_tier, and component scores

In [36]:
def score_new_complaint(complaint_text, sentiment_label='neutral', model_confidence=0.5, created_date=None):
    
    if created_date is None:
        created_date = pd.Timestamp.now()
    
    # Create a fake row with the data
    fake_row = {
        'combined_text': complaint_text,
        'clean_text': complaint_text,
        'sentiment_label': sentiment_label,
        'model_confidence': model_confidence,
        'Created Date': created_date
    }
    
    scores = calculate_urgency_score(fake_row)
    priority = get_priority_tier(scores['urgency_score'])
    
    return {
        'complaint_text': complaint_text,
        'sentiment_label': sentiment_label,
        'model_confidence': model_confidence,
        'created_date': str(created_date),
        'urgency_score': round(scores['urgency_score'], 2),
        'priority_tier': priority,
        'sla': PRIORITY_TIERS[priority]['sla'],
        'component_scores': {
            'sentiment': round(scores['sentiment_score'], 2),
            'keywords': round(scores['keyword_score'], 2),
            'confidence': round(scores['confidence_score'], 2),
            'recency': round(scores['recency_score'], 2)
        }
    }


# Test the live scoring function
print('LIVE SCORING FUNCTION TEST')

# Test cases
test_cases = [
    {
        'text': 'There is a fire in the building and people are trapped inside',
        'sentiment': 'critical',
        'confidence': 0.95
    },
    {
        'text': 'The garbage bin has not been collected for a week',
        'sentiment': 'neutral',
        'confidence': 0.70
    },
    {
        'text': 'There is an emergency with a live wire on the street',
        'sentiment': 'negative',
        'confidence': 0.88
    }
]

for i, test in enumerate(test_cases, 1):
    result = score_new_complaint(
        test['text'],
        test['sentiment'],
        test['confidence']
    )
    
    print(f'\nTest Case #{i}:')
    print(f"  Text: {result['complaint_text'][:60]}...")
    print(f"  Sentiment: {result['sentiment_label']}, Confidence: {result['model_confidence']}")
    print(f"  Urgency Score: {result['urgency_score']}")
    print(f"  Priority: {result['priority_tier']} (SLA: {result['sla']})")
    print(f"  Component Scores: {result['component_scores']}")

print('\n✅ Live scoring function works correctly')

LIVE SCORING FUNCTION TEST

Test Case #1:
  Text: There is a fire in the building and people are trapped insid...
  Sentiment: critical, Confidence: 0.95
  Urgency Score: 84.0
  Priority: P1 (SLA: 2 hours)
  Component Scores: {'sentiment': 100, 'keywords': 40, 'confidence': 95.0, 'recency': 100.0}

Test Case #2:
  Text: The garbage bin has not been collected for a week...
  Sentiment: neutral, Confidence: 0.7
  Urgency Score: 39.5
  Priority: P4 (SLA: 7 days)
  Component Scores: {'sentiment': 20, 'keywords': 10, 'confidence': 70.0, 'recency': 100.0}

Test Case #3:
  Text: There is an emergency with a live wire on the street...
  Sentiment: negative, Confidence: 0.88
  Urgency Score: 62.85
  Priority: P2 (SLA: 24 hours)
  Component Scores: {'sentiment': 60, 'keywords': 25, 'confidence': 88.0, 'recency': 100.0}

✅ Live scoring function works correctly


## Final Summary Checklist

In [37]:
print('OUTPUT FILES GENERATED:')
print(f'  📊 {output_csv.name}')
print(f'     Location: {output_csv}')
print(f'     Size: {output_csv.stat().st_size / 1024 / 1024:.2f} MB')
print(f'     Rows: {len(df_scored_sorted):,}')
print()
print(f'  ⚙️  {config_path.name}')
print(f'     Location: {config_path}')
print(f'     Contains: Weights, Thresholds, Keywords, Stats')


OUTPUT FILES GENERATED:
  📊 grievance_with_urgency.csv
     Location: ..\outputs\grievance_with_urgency.csv
     Size: 6.89 MB
     Rows: 12,387

  ⚙️  urgency_config.json
     Location: ..\outputs\urgency_config.json
     Contains: Weights, Thresholds, Keywords, Stats
